# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagnik556/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
import duckdb
con = duckdb.connect()

# Use your HF_TOKEN secret (never paste the token directly in a cell)
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# See what tables/files actually exist
# Display the first 5 rows
con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 5").show()
# Display the schema of the full relation
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 1").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬────────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │  gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_cl

In [27]:
import pandas as pd
schema = con.sql(f""" DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 1
""").df()
pd.set_option('display.max_rows', None)
print(schema['column_name'].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. Unit of analysis + time window


**One row = one what, over which dates?**

One row = one piece of content, for one client, on one calendar day
(content_hash_id × client_hash_id × report_date).

Time window: a mid-panel development month, month='2026-03' (using the
table's own `month` column). The final month (June 2026, the `_sample` table)
is a sealed test window — I will not build label logic against it.

Claim verified below in Section 3.

## 2. Fields: feature / label / context / excluded
**Feature** (trailing 7-day windows only, so they're knowable before today):
- sessions_prior_7d_avg (from ga4_sessions)
- gsc_position_prior_7d_avg (from gsc_avg_position)
- scroll_events_prior_7d_avg (from scroll_events)
- ai_traffic_share_prior_7d (from sessions_ai / ga4_sessions)
- engagement_rate_prior_7d (from ga4_engaged_sessions / ga4_sessions)

**Label:** engagement_rate_today = ga4_engaged_sessions / ga4_sessions
(today's value), later inverted via percentile rank into a risk score.

**Context** (used for identity/filtering, not modeling):
content_hash_id, client_hash_id, report_date, client_has_gsc, client_has_ga4,
gsc_data_available, ga4_data_available.

**Excluded:**
- gsc_impressions, gsc_clicks, gsc_sum_position — these measure search
  discovery, not post-landing reader behavior, which is out of scope for
  engagement_fix.
- sessions_organic/direct/referral/social/paid, ai_chatgpt/perplexity/gemini/
  copilot/claude/meta/other — individual channel/AI-source breakdowns add
  detail beyond what a content-day engagement score needs; I'm using the
  already-aggregated sessions_ai / ga4_sessions ratio instead of the per-source
  columns.

## 3. Verify it with queries (grain, counts, missing values, windows)

**Windows finding:** [fill in once run — e.g. "X of 413,966 rows had no prior
7-day history and were dropped. The leaky feature correlated at ~1.0 with the
label, since it IS the label computed on the same day. The honest prior-7d
feature correlated at [real number] — a real but imperfect signal, confirming
the leak was caught and removed."]

In [28]:


# Grain check
con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE {month_filter}
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 10
""").show()
# -> 0 rows: grain claim in Section 1 confirmed

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────┬─────────────┬───────────┐
│ content_hash_id │ client_hash_id │ report_date │ row_count │
│     varchar     │    varchar     │    date     │   int64   │
├─────────────────┴────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘



In [29]:
# Counts + date span
con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as earliest, MAX(report_date) as latest
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE {month_filter}
""").show()


┌────────────┬────────────┬────────────┐
│ total_rows │  earliest  │   latest   │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘



In [30]:
# Missing values — GA4 availability
con.sql(f"""
    SELECT COUNT(*) as available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE {month_filter} AND ga4_data_available IS TRUE
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         413966 │
└────────────────┘



In [31]:
# Windows — build the 7-day trailing features, then demonstrate the leak trap
features_query = f"""
    SELECT
        content_hash_id, client_hash_id, report_date,
        AVG(ga4_sessions) OVER (PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS sessions_prior_7d_avg,
        AVG(gsc_avg_position) OVER (PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS gsc_position_prior_7d_avg,
        AVG(scroll_events) OVER (PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS scroll_events_prior_7d_avg,
        AVG(CASE WHEN ga4_sessions > 0 THEN sessions_ai * 1.0 / ga4_sessions END)
            OVER (PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS ai_traffic_share_prior_7d,
        AVG(CASE WHEN ga4_sessions > 0 THEN ga4_engaged_sessions * 1.0 / ga4_sessions END)
            OVER (PARTITION BY content_hash_id ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS engagement_rate_prior_7d,
        CASE WHEN ga4_sessions > 0 THEN ga4_engaged_sessions * 1.0 / ga4_sessions END AS engagement_rate_today
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE {month_filter} AND ga4_data_available IS TRUE
"""
features_df = con.sql(features_query).df()

# check target distribution isn't degenerate
print(features_df['engagement_rate_today'].describe())

# drop rows with no prior history (window has nothing to average yet)
features_clean = features_df.dropna(subset=['engagement_rate_prior_7d']).copy()
print(f"\nRows with full 7-day history: {len(features_clean)} of {len(features_df)}")

# the trap: add a label-derived "feature" and watch the score jump
features_clean['leaky_feature'] = features_clean['engagement_rate_today']
print("\nWith leak:")
print(features_clean[['leaky_feature', 'engagement_rate_today']].corr())

print("\nHonest feature only:")
print(features_clean[['engagement_rate_prior_7d', 'engagement_rate_today']].corr())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

count    410335.000000
mean          0.035102
std           0.161971
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: engagement_rate_today, dtype: float64

Rows with full 7-day history: 322831 of 413966

With leak:
                       leaky_feature  engagement_rate_today
leaky_feature                    1.0                    1.0
engagement_rate_today            1.0                    1.0

Honest feature only:
                          engagement_rate_prior_7d  engagement_rate_today
engagement_rate_prior_7d                  1.000000               0.055497
engagement_rate_today                     0.055497               1.000000


In [32]:
# --- Section 3: five features, trailing 7-day windows only ---

features_query = f"""
    SELECT
        content_hash_id, client_hash_id, report_date,
        AVG(ga4_sessions) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS sessions_prior_7d_avg,
        AVG(gsc_avg_position) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS gsc_position_prior_7d_avg,
        AVG(scroll_events) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS scroll_events_prior_7d_avg,
        AVG(CASE WHEN ga4_sessions > 0 THEN sessions_ai * 1.0 / ga4_sessions END) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS ai_traffic_share_prior_7d,
        AVG(CASE WHEN ga4_sessions > 0 THEN ga4_engaged_sessions * 1.0 / ga4_sessions END) OVER (
            PARTITION BY content_hash_id
            ORDER BY report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS engagement_rate_prior_7d,
        CASE WHEN ga4_sessions > 0 THEN ga4_engaged_sessions * 1.0 / ga4_sessions END AS engagement_rate_today
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE {month_filter} AND ga4_data_available IS TRUE
"""
features_df = con.sql(features_query).df()
features_df.head()

,content_hash_id,client_hash_id,report_date,sessions_prior_7d_avg,gsc_position_prior_7d_avg,scroll_events_prior_7d_avg,ai_traffic_share_prior_7d,engagement_rate_prior_7d,engagement_rate_today
0,content_000612ace4167db9,client_73cda7b4e4f265ea,2026-03-24,NaN,NaN,NaN,NaN,NaN,0.0
1,content_00106e4a3275015f,client_ba65e80a1116ae41,2026-03-22,NaN,NaN,NaN,NaN,NaN,0.0
2,content_001901efa4524f68,client_3ffa76342f366962,2026-03-21,NaN,NaN,NaN,NaN,NaN,0.0
3,content_001ddf6f43eeaf75,client_fef1a8f436438636,2026-03-07,NaN,NaN,NaN,NaN,NaN,0.0
4,content_001ddf6f43eeaf75,client_fef1a8f436438636,2026-03-08,1.0,10.084746,0.0,0.0,0.0,0.0


## 4. Data limits

**What can this data never tell you?**

- **Unbalanced history:** content_hash_ids don't all have the same tracking
  history within the month — some only appear partway through March, so their
  trailing 7-day windows are incomplete (NaN) for their first several days.
  Rows without full history were excluded from the leak/window check above.

- **GSC-only early rows:** client_has_gsc and client_has_ga4 are independent
  flags — only ~4.2% of March rows have GA4 data at all. A large share of
  rows have GSC signal but no GA4 signal, meaning engagement_fix's label
  simply can't be computed for most of the table, not just weakly measured.

- **Window overlaps:** each day's trailing 7-day window shares 6 of its 7
  days with the next day's window for the same content. Consecutive daily
  rows for the same content_hash_id are therefore highly autocorrelated, not
  independent observations — evaluating performance day-by-day would overstate
  stability compared to genuinely independent samples.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.